# Search-05b : Natural Evolution Strategies en C#

**Navigation** : [Version Python](Search-05b-NaturalEvolutionStrategies.ipynb) · [Fondations](README.md)

**Objectifs** : dériver les gradients de recherche et naturels, comparer xNES, SNES et CMA-ES sur un ellipsoïde tourné, et comprendre le lien au premier ordre. **Prérequis** : probabilités, matrices et optimisation. **Durée** : 2 h.

## Préparation

Depuis la racine du dépôt, initialiser le sous-module et construire la dépendance exacte avant d'ouvrir ce carnet :

```bash
git submodule update --init --recursive MyIA.AI.Notebooks/Search/MetaGeneticSharp
dotnet build MyIA.AI.Notebooks/Search/MetaGeneticSharp/src/MetaGeneticSharp.Domain/MetaGeneticSharp.Domain.csproj -c Release
```

.NET 9 et le noyau .NET Interactive sont requis. Le gitlink fixe MetaGeneticSharp à `824c54e142efa6ae434890fc2ae83a3515aab1e3` et son sous-module GeneticSharp ; aucune clé ni GPU n'est nécessaire. Exécuter avec le répertoire courant de ce carnet. Les graphiques sont des SVG statiques.

In [1]:
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/GeneticSharp.Domain.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/MetaGeneticSharp.Domain.dll"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Net;
using System.Globalization;
using Microsoft.DotNet.Interactive.Formatting;
using MetaGeneticSharp;
using M = MetaGeneticSharp.NaturalEvolutionStrategyMath;
Console.WriteLine($"MetaGeneticSharp {typeof(NesDistribution).Assembly.GetName().Version}, GeneticSharp {typeof(GeneticSharp.IRandomization).Assembly.GetName().Version}");

MetaGeneticSharp 1.0.0.0, GeneticSharp 3.1.4.0


## 1. Gradient de recherche et gradient naturel

Pour maximiser $E[-X^2]$, le gradient ordinaire estime les dérivées du log de la densité. Une mise à jour additive de $\sigma$ peut sortir du domaine positif. Le gradient naturel emploie des rangs et multiplie l'échelle par une exponentielle. Les médianes ne concernent que les trajectoires encore valides ; le nombre d'échecs est présenté séparément.

Les deux méthodes diffèrent simultanément par la géométrie, les rangs et les taux : le gradient ordinaire utilise $\eta=0{,}01$, le naturel $\eta_\mu=1$ et $\eta_\sigma=0{,}6$ en dimension 1. Cette comparaison ne maintient donc pas le pas constant.

In [2]:
double Normal(Random r) => Math.Sqrt(-2*Math.Log(1-r.NextDouble()))*Math.Cos(2*Math.PI*r.NextDouble());
double[] Utilities(int n) { var a=Enumerable.Range(1,n).Select(k=>Math.Max(0,Math.Log(n/2.0+1)-Math.Log(k))).ToArray(); var sum=a.Sum(); return a.Select(x=>x/sum-1.0/n).ToArray(); }
int Lambda(int d)=>4+(int)Math.Floor(3*Math.Log(d));
(double[],double[]) OneD(int seed, bool natural) {
 int count=natural?200:1000, n=10; var mu=1.0; var sigma=1.0; var r=new Random(seed); var ms=Enumerable.Repeat(double.NaN,count).ToArray(); var ss=(double[])ms.Clone(); var u=Utilities(n);
 for(int g=0;g<count;g++) { var z=Enumerable.Range(0,n).Select(_=>Normal(r)).ToArray(); var x=z.Select(v=>mu+sigma*v).ToArray();
  if(natural) { var ranked=Enumerable.Range(0,n).OrderBy(i=>x[i]*x[i]).ToArray(); double gm=0,gs=0; for(int k=0;k<n;k++){double v=z[ranked[k]];gm+=u[k]*v;gs+=u[k]*(v*v-1);} mu+=sigma*gm; sigma*=Math.Exp(M.DefaultScaleLearningRate(NesCovarianceModel.Separable,1)*gs/2); }
  else { double gm=0,gs=0;for(int k=0;k<n;k++){double f=-x[k]*x[k];gm+=f*z[k]/sigma;gs+=f*(z[k]*z[k]-1)/sigma;} mu+=.01*gm/n;sigma+=.01*gs/n; }
  if(!double.IsFinite(mu)||!double.IsFinite(sigma)||sigma<=0) break; ms[g]=Math.Abs(mu); ss[g]=sigma;
 } return (ms,ss);
}
double Median(IEnumerable<double> xs) { var a=xs.Where(double.IsFinite).OrderBy(x=>x).ToArray(); return a.Length==0?double.NaN:(a[(a.Length-1)/2]+a[a.Length/2])/2; }
var simple=Enumerable.Range(0,300).Select(i=>OneD(i,false)).ToArray(); var naturel=Enumerable.Range(0,300).Select(i=>OneD(i,true)).ToArray();
foreach(var item in new[]{("simple",simple), ("naturel",naturel)}) { Console.WriteLine(item.Item1); foreach(var g in new[]{0,10,50,100,item.Item2[0].Item1.Length-1}.Distinct()) Console.WriteLine($"g={g}, échecs={item.Item2.Count(r=>!double.IsFinite(r.Item2[g]))}/300, médiane |mu|={Median(item.Item2.Select(r=>r.Item1[g])):G4}, sigma={Median(item.Item2.Select(r=>r.Item2[g])):G4}"); }

simple


g=0, échecs=0/300, médiane |mu|=0.9847, sigma=0.9931


g=10, échecs=0/300, médiane |mu|=0.8016, sigma=0.813


g=50, échecs=0/300, médiane |mu|=0.3562, sigma=0.3643


g=100, échecs=0/300, médiane |mu|=0.1295, sigma=0.1337


g=999, échecs=97/300, médiane |mu|=1.559E-09, sigma=1.58E-09


... (2 outputs left out)

g=10, échecs=0/300, médiane |mu|=0.0222, sigma=0.1032


g=50, échecs=0/300, médiane |mu|=2.83E-06, sigma=1.211E-05


g=100, échecs=0/300, médiane |mu|=3.434E-11, sigma=1.479E-10


g=199, échecs=0/300, médiane |mu|=4.655E-21, sigma=2.237E-20


Les échecs correspondent à une distribution invalide, pas à une valeur optimale. La mise en forme par rang rend le pas naturel insensible à toute transformation strictement croissante de l'objectif, sous réserve de conserver les mêmes classements.

In [3]:
var u10=Utilities(10); Console.WriteLine("Utilités : "+string.Join(", ",u10.Select(x=>x.ToString("F4")))+ $" ; somme={u10.Sum():G3}");
if(Math.Abs(u10.Sum())>1e-12) throw new Exception("Somme des utilités incorrecte");
foreach(int d in new[]{1,2,5,10,20,40}) Console.WriteLine($"d={d}, lambda={Lambda(d)}, xNES={M.DefaultShapeLearningRate(d):F5}, SNES={M.DefaultScaleLearningRate(NesCovarianceModel.Separable,d):F5}");
foreach(var name in new[]{"ExponentialNaturalEvolutionStrategy","SeparableNaturalEvolutionStrategy"}) Console.WriteLine($"Factory {name} : {MetaHeuristicsService.CreateMetaHeuristicByName(name).GetType().Name}");

Utilités : 0.3295, 0.1634, 0.0662, -0.0028, -0.0563, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000 ; somme=-1.94E-16


d=1, lambda=4, xNES=1.80000, SNES=0.60000


d=2, lambda=6, xNES=0.78343, SNES=0.52229


d=5, lambda=8, xNES=0.24737, SNES=0.41228


d=10, lambda=10, xNES=0.10061, SNES=0.33536


d=20, lambda=12, xNES=0.04022, SNES=0.26814


d=40, lambda=15, xNES=0.01586, SNES=0.21152


Factory ExponentialNaturalEvolutionStrategy : NaturalEvolutionStrategyMetaHeuristic


Factory SeparableNaturalEvolutionStrategy : NaturalEvolutionStrategyMetaHeuristic


Les utilités centrées et les taux sont ceux de la table 1. Le moteur réel reçoit des échantillons triés du meilleur au moins bon. Le calcul explicite suivant est un oracle indépendant pour un pas SNES.

In [4]:
var anchors=new List<double[]>{new[]{1d,0d},new[]{-1d,0d},new[]{0d,1d},new[]{0d,-1d}};
var ranked=new List<double[]>{new[]{.1,.1},new[]{.3,.1},new[]{-.4,-.2},new[]{1d,1d}};
var init=NesDistribution.Fit(NesCovarianceModel.Separable,anchors); var eta=M.DefaultScaleLearningRate(NesCovarianceModel.Separable,2); var step=init.Update(ranked,1,eta,M.DefaultShapeLearningRate(2)); var weights=Utilities(ranked.Count);
for(int j=0;j<2;j++){double gm=0,gs=0;for(int k=0;k<ranked.Count;k++){double z=(ranked[k][j]-init.Mean[j])/init.Scales[j];gm+=weights[k]*z;gs+=weights[k]*(z*z-1);} double mean=init.Mean[j]+init.Scales[j]*gm, scale=init.Scales[j]*Math.Exp(eta*gs/2); Console.WriteLine($"coord {j}: écart moyenne={Math.Abs(mean-step.Mean[j]):G3}, écart échelle={Math.Abs(scale-step.Scales[j]):G3}"); if(Math.Abs(mean-step.Mean[j])+Math.Abs(scale-step.Scales[j])>1e-12) throw new Exception("Oracle SNES");}
var full=NesDistribution.Fit(NesCovarianceModel.Full,anchors);double rate=M.DefaultShapeLearningRate(2);var fullStep=full.Update(ranked,1,rate,rate);var gm2=new double[2];var matrix=new double[2,2];for(int k=0;k<ranked.Count;k++){var z=full.ToNatural(ranked[k]);for(int a=0;a<2;a++){gm2[a]+=weights[k]*z[a];for(int b=0;b<2;b++)matrix[a,b]+=weights[k]*(z[a]*z[b]-(a==b?1:0));}}double trace=(matrix[0,0]+matrix[1,1])/2;var shapeArg=new double[2,2];for(int a=0;a<2;a++)for(int b=0;b<2;b++)shapeArg[a,b]=rate/2*(matrix[a,b]-(a==b?trace:0));var expectShape=M.MatrixExponential(shapeArg);double maxError=0;for(int a=0;a<2;a++){maxError=Math.Max(maxError,Math.Abs(fullStep.Mean[a]-(full.Mean[a]+full.Sigma*gm2[a])));for(int b=0;b<2;b++)maxError=Math.Max(maxError,Math.Abs(fullStep.Shape[a,b]-expectShape[a,b]));}maxError=Math.Max(maxError,Math.Abs(fullStep.Sigma-full.Sigma*Math.Exp(rate*trace/2)));Console.WriteLine($"Oracle explicite xNES : écart maximal={maxError:G3}");if(maxError>1e-12)throw new Exception("Oracle xNES");

coord 0: écart moyenne=0, écart échelle=0


coord 1: écart moyenne=0, écart échelle=0


Oracle explicite xNES : écart maximal=0


### Exercice 1 — Utilités linéaires

Construire des poids affines strictement décroissants en rang, de somme nulle, normalisés par $u_1-u_\lambda=1$. Puis compléter la fonction de comparaison : elle doit implémenter un pas pédagogique avec ces poids et les comparer aux poids logarithmiques sur les mêmes tirages. `NesDistribution.Update` utilise ses propres utilités et ne permet pas d'injecter des poids ; ce chemin personnalisé est donc à programmer explicitement.

In [5]:
double[] LinearUtilities(int lambda) { // TODO : poids affines décroissants, centrés, amplitude 1
 return null;
}
double? CompareCustomUtilities(double[] weights, int seed) { // TODO : pas pédagogique utilisant ces poids, mêmes tirages que la référence
 return null;
}
var linear=LinearUtilities(10);
if(linear==null) Console.WriteLine("Exercice 1 à compléter : utilités linéaires");
else {bool valid=linear.Length==10 && linear.All(double.IsFinite) && Math.Abs(linear.Sum())<1e-10 && Math.Abs(linear[0]-linear[9]-1)<1e-10 && Enumerable.Range(1,8).All(i=>Math.Abs(linear[i-1]-2*linear[i]+linear[i+1])<1e-10);Console.WriteLine($"Longueur, normalisation, centrage et secondes différences nulles : {valid}");if(!valid)throw new Exception("Corriger les utilités linéaires");var comparison=CompareCustomUtilities(linear,42);if(comparison==null)Console.WriteLine("Comparaison personnalisée à compléter");else if(!double.IsFinite(comparison.Value))throw new Exception("Comparaison non finie");else Console.WriteLine($"Écart mesuré : {comparison}");}

Exercice 1 à compléter : utilités linéaires


## 2. Banc d'optimisation

On minimise un ellipsoïde translaté : son optimum n'est pas le point de départ $(3,\ldots,3)$. Les points symétriques imposent exactement moyenne 3 et écart-type 1 à `Fit`. La rotation est construite par rotations planes avec la graine 2014 ; ce n'est pas la matrice NumPy de la version Python. Le compteur comprend toutes les évaluations et le dernier lot peut dépasser le budget nominal.

In [6]:
double[,] Rotation(int d,int seed){var r=new Random(seed);var q=M.Identity(d);for(int k=0;k<d*4;k++){int a=r.Next(d),b=(a+1+r.Next(d-1))%d;double t=2*Math.PI*r.NextDouble(),c=Math.Cos(t),s=Math.Sin(t);for(int i=0;i<d;i++){double x=q[i,a],y=q[i,b];q[i,a]=c*x-s*y;q[i,b]=s*x+c*y;}}return q;}
double Objective(double[] x,double[,] q,bool rotated){int d=x.Length;double result=0;for(int i=0;i<d;i++){double y=0;for(int j=0;j<d;j++)y+=(rotated?q[j,i]:(i==j?1:0))*(x[j]-.7);result+=Math.Pow(1e6,(double)i/(d-1))*y*y;}return result;}
List<double[]> Anchors(int d){var a=new List<double[]>();for(int i=0;i<d;i++)foreach(int sign in new[]{-1,1}){var x=Enumerable.Repeat(3d,d).ToArray();x[i]+=sign*Math.Sqrt(d);a.Add(x);}return a;}
record Run(int Evals,double Best,List<(int,double)> Trace);
Run Optimize(string method,int d,int seed,int budget,Func<double[],double> f,double target=1e-8){
 var r=new Random(seed); int n=Lambda(d),evals=0; double best=double.PositiveInfinity; var trace=new List<(int,double)>(); var model=method=="SNES"?NesCovarianceModel.Separable:NesCovarianceModel.Full; var dist=NesDistribution.Fit(model,Anchors(d)); var cma=CmaEsDistribution.Create(Enumerable.Repeat(3d,d).ToArray(),1,M.Identity(d));
 while(evals<budget){var samples=new List<(double[],double)>();for(int k=0;k<n;k++){var z=Enumerable.Range(0,d).Select(_=>Normal(r)).ToArray();var x=method=="CMA-ES"?cma.FromNatural(z):dist.FromNatural(z);samples.Add((x,f(x)));}evals+=n;samples=samples.OrderBy(p=>p.Item2).ToList();best=Math.Min(best,samples[0].Item2);trace.Add((evals,best));if(best<=target)break;var xs=samples.Select(p=>p.Item1).ToList();if(method=="CMA-ES") cma=cma.UpdateWithFitness(xs,samples.Select(p=>-p.Item2).ToArray()); else dist=dist.Update(xs,1,M.DefaultScaleLearningRate(model,d),M.DefaultShapeLearningRate(d)); }
 return new Run(evals,best,trace);
}
var rot=Rotation(10,2014);double orth=0;for(int i=0;i<10;i++)for(int j=0;j<10;j++){double v=0;for(int k=0;k<10;k++)v+=rot[k,i]*rot[k,j];orth=Math.Max(orth,Math.Abs(v-(i==j?1:0)));}Console.WriteLine($"Erreur orthogonalité maximale={orth:G3} ; condition théorique préservée=1e6 ; f(départ)={Objective(Enumerable.Repeat(3d,10).ToArray(),rot,true):G5}");if(orth>1e-12)throw new Exception("Rotation non orthogonale");
var a10=NesDistribution.Fit(NesCovarianceModel.Full,Anchors(10)); Console.WriteLine($"Initialisation : moyenne={a10.Mean[0]}, sigma={a10.Sigma}");

Erreur orthogonalité maximale=6.66E-16 ; condition théorique préservée=1e6 ; f(départ)=4.4952E+06


Initialisation : moyenne=3, sigma=1


La transformation monotone conserve les rangs. Les deux runs ci-dessous utilisent le même flux aléatoire et le même budget ; l'écart est calculé après retour à l'échelle initiale.

Pour contraster, on compare aussi le gradient ordinaire sur cinq graines : les deux variantes reçoivent les mêmes tirages mais les amplitudes de fitness changent. Un échec signifie que la distribution devient invalide.

In [7]:
var q3=Rotation(3,2014); Func<double[],double> f3=x=>Objective(x,q3,true); Func<double[],double> g3=x=>Math.Pow(Math.Log(1+f3(x)),3);var fRun=Optimize("xNES",3,5,900,f3,0);var gRun=Optimize("xNES",3,5,900,g3,0);double diff=fRun.Trace.Zip(gRun.Trace,(a,b)=>Math.Abs(a.Item2-(Math.Exp(Math.Pow(b.Item2,1.0/3))-1))).Max();Console.WriteLine($"f : {fRun.Evals} évaluations, g : {gRun.Evals} évaluations ; écart maximal ramené à f={diff:G5}");
double relative=fRun.Trace.Zip(gRun.Trace,(a,b)=>Math.Abs(a.Item2-(Math.Exp(Math.Pow(b.Item2,1.0/3))-1))/Math.Max(1,Math.Abs(a.Item2))).Max();if(fRun.Evals!=gRun.Evals||relative>1e-10)throw new Exception("Invariance monotone non vérifiée");Console.WriteLine($"Écart relatif maximal={relative:G4} ; invariance vérifiée");
(double? value,int generations) OrdinaryTransformed(int seed,bool logFitness){var random=new Random(seed);double mu=1,sigma=1;for(int generation=0;generation<300;generation++){double gm=0,gs=0;for(int k=0;k<10;k++){double z=Normal(random),x=mu+sigma*z,v=x*x;double fitness=-(logFitness?Math.Log(1+v):v);gm+=fitness*z/sigma;gs+=fitness*(z*z-1)/sigma;}double newMu=mu+.01*gm/10,newSigma=sigma+.01*gs/10;if(!double.IsFinite(newMu)||!double.IsFinite(newSigma)||newSigma<=0)return (null,generation+1);mu=newMu;sigma=newSigma;}return (Math.Abs(mu),300);}
foreach(int seed in Enumerable.Range(0,5)){var raw=OrdinaryTransformed(seed,false);var log=OrdinaryTransformed(seed,true);string Format((double? value,int generations) x)=>x.value.HasValue?$"{x.value.Value:G6}":$"échec à la génération {x.generations}";Console.WriteLine($"graine {seed} : |mu| brut={Format(raw)}, log(1+x²)={Format(log)}");}

f : 903 évaluations, g : 903 évaluations ; écart maximal ramené à f=3.7253E-08


Écart relatif maximal=2.8E-15 ; invariance vérifiée


graine 0 : |mu| brut=0.00180763, log(1+x²)=0.008488


graine 1 : |mu| brut=0.00212197, log(1+x²)=0.0116477


graine 2 : |mu| brut=0.00214395, log(1+x²)=0.0127094


graine 3 : |mu| brut=0.00193717, log(1+x²)=0.00990372


graine 4 : |mu| brut=0.00311401, log(1+x²)=0.018872


Les deux fonctions classent les points de la même façon, mais le gradient ordinaire pondère les échantillons par leurs valeurs de fitness : ses résultats changent ici avec la transformation. Cinq graines illustrent cet effet sans établir une loi générale. Pour xNES, le dénominateur `max(1, |f|)` mélange erreur absolue et relative près de zéro ; le test porte sur le meilleur objectif cumulé, pas sur toutes les coordonnées.

### Exercice 2 — Coût en dimension

Sur la sphère $f(x)=\sum x_i^2$, comparer xNES et SNES en dimensions 10, 40 et 100, pour la cible $10^{-8}$ et un budget nominal de 40 000 évaluations, par lots entiers. Mesurer pour chaque couple le nombre d'évaluations et le temps ; indiquer si la cible est atteinte. Les mesures restent à réaliser, sans supposer un classement des temps.

In [8]:
int[] dimensions={10,40,100}; string[] methods={"xNES","SNES"}; const int exerciseBudget=40000; const double exerciseTarget=1e-8;
record CostResult(int Evaluations,double Seconds,double Best);
Dictionary<(int,string),CostResult> MeasureCosts() { // TODO : mesurer les six couples jusqu'à la cible ou au budget
 return null;
}
var costs=MeasureCosts();
if(costs==null) Console.WriteLine("Exercice 2 à compléter : évaluations, meilleur objectif et temps des six couples");
else {bool valid=costs.Count==6 && dimensions.All(d=>methods.All(m=>costs.TryGetValue((d,m),out var v)&&v!=null && v.Evaluations>0 && v.Evaluations%Lambda(d)==0 && v.Evaluations<=exerciseBudget+Lambda(d)-1 && double.IsFinite(v.Seconds)&&v.Seconds>=0 && double.IsFinite(v.Best)&&v.Best>=0 && (v.Best<=exerciseTarget || v.Evaluations>=exerciseBudget)));Console.WriteLine($"Six mesures structurées et budgets cohérents : {valid}");if(!valid)throw new Exception("Corriger les mesures");foreach(var entry in costs)Console.WriteLine($"{entry.Key} : {entry.Value.Evaluations} évaluations, {entry.Value.Seconds:G4} s, cible atteinte = {entry.Value.Best<=exerciseTarget}");}

Exercice 2 à compléter : évaluations, meilleur objectif et temps des six couples


Le banc utilise quatre graines pour chaque méthode et chaque orientation. Le moteur CMA-ES maximise une fitness : on lui transmet donc $-f$ après tri croissant de $f$. Il s'agit de l'implémentation MGS, pas de la bibliothèque Python `cma` ; les RNG diffèrent aussi. Les résultats individuels ne sont pas supposés identiques.

In [9]:
var results=new Dictionary<string,List<Run>>();foreach(bool rotated in new[]{false,true}) foreach(string method in new[]{"xNES","SNES","CMA-ES"}) {var label=(rotated?"tourné":"aligné")+" "+method;var runs=new List<Run>();foreach(int seed in new[]{0,1,7,42})runs.Add(Optimize(method,10,seed,40000,x=>Objective(x,rot,rotated)));results[label]=runs;Console.WriteLine($"{label}: succès={runs.Count(r=>r.Best<=1e-8)}/4, évaluations min/médiane/max={runs.Min(r=>r.Evals)}/{Median(runs.Select(r=>(double)r.Evals))}/{runs.Max(r=>r.Evals)}, meilleur f={runs.Min(r=>r.Best):G4}");}

aligné xNES: succès=4/4, évaluations min/médiane/max=8850/8960/9110, meilleur f=7.374E-09


aligné SNES: succès=4/4, évaluations min/médiane/max=2630/2720/2810, meilleur f=6.614E-09


aligné CMA-ES: succès=4/4, évaluations min/médiane/max=5770/5990/6270, meilleur f=4.335E-09


tourné xNES: succès=4/4, évaluations min/médiane/max=8820/8870/8910, meilleur f=6.897E-09


tourné SNES: succès=0/4, évaluations min/médiane/max=40000/40000/40000, meilleur f=412.4


tourné CMA-ES: succès=4/4, évaluations min/médiane/max=5770/5955/6030, meilleur f=5.043E-09


Les comptes indiquent les évaluations réellement consommées, y compris le dernier lot. Le classement empirique sur cet exemple ne prouve pas l'équivalence par rotation de manière générale.

In [10]:
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((p,w)=>w.Write(p.Content),"text/html");
string Esc(string s)=>WebUtility.HtmlEncode(s);
string F(double x)=>x.ToString("F2",CultureInfo.InvariantCulture);
string Figure(){var sb=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='900' height='400' viewBox='0 0 900 400'><rect width='900' height='400' fill='white'/>");string[] colors={"#276fbf","#c14953","#21856b"};int panel=0;foreach(string orientation in new[]{"aligné","tourné"}){int left=55+panel*440;sb.Append($"<text x='{left}' y='25'>{Esc(orientation)}</text><path d='M{left} 40 V340 H{left+390}' fill='none' stroke='black'/><text x='{left+100}' y='390'>évaluations effectives</text><text x='{left}' y='355'>0</text><text x='{left+340}' y='355'>40000</text><text x='{left}' y='38'>log10 f</text>");foreach(int tick in new[]{-8,-4,0,4,8}) {double y=40+300*(8-tick)/18.0;sb.Append($"<text x='{left-30}' y='{F(y+4)}'>{tick}</text><path d='M{left-4} {F(y)} H{left}' stroke='black'/>");}foreach(int tick in new[]{10000,20000,30000}) {double x=left+390.0*tick/40000;sb.Append($"<text x='{F(x-14)}' y='355'>{tick/1000}k</text><path d='M{F(x)} 340 v4' stroke='black'/>");}int m=0;foreach(string method in new[]{"xNES","SNES","CMA-ES"}){var run=results[orientation+" "+method].OrderBy(r=>r.Evals).ThenBy(r=>r.Best).ElementAt(1);var pts=run.Trace.Where((p,i)=>i%10==0||i==run.Trace.Count-1).Select(p=>$"{F(left+390.0*p.Item1/40010)} {F(40+300*(8-Math.Clamp(Math.Log10(Math.Max(p.Item2,1e-12)),-10,8))/18)}");sb.Append($"<polyline fill='none' stroke='{colors[m]}' points='{string.Join(" ",pts)}'/><text x='{left+90*m}' y='375' fill='{colors[m]}'>{Esc(method)}</text>");m++;}sb.Append($"<path d='M{left} 306.67 H{left+390}' stroke='#777' stroke-dasharray='4 3'/><text x='{left+300}' y='302'>cible 1e-8</text>");panel++;}return sb.Append("</svg>").ToString();}
display(new PlotSvg(Figure()));
string GradientFigure(){var sb=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='1040' height='420' viewBox='0 0 1040 420'><rect width='1040' height='420' fill='white'/>");int panel=0;foreach(var data in new[]{simple,naturel}){int left=65+panel*510, count=data[0].Item1.Length, width=370;sb.Append($"<text x='{left}' y='22'>{(panel==0?"gradient simple (1000 générations)":"gradient naturel (200 générations)")}</text><path d='M{left} 45 V335 H{left+width} V45' fill='none' stroke='black'/><text x='{left}' y='38'>log10 valeur</text><text x='{left+width-12}' y='38' fill='#555'>échecs %</text><text x='{left+120}' y='400'>génération</text>");foreach(int tick in new[]{0,-5,-10,-15,-20}){double y=45+290*(-tick)/22.0;sb.Append($"<text x='{left-30}' y='{F(y+4)}'>{tick}</text><path d='M{left-4} {F(y)} H{left}' stroke='black'/>");}foreach(int tick in new[]{0,25,50,75,100}){double y=335-290*tick/100.0;sb.Append($"<text x='{left+width+8}' y='{F(y+4)}' fill='#555'>{tick}</text><path d='M{left+width} {F(y)} h4' stroke='#555'/>");}foreach(double frac in new[]{0d,.25,.5,.75,1d}){double x=left+width*frac;int tick=(int)Math.Round((count-1)*frac);sb.Append($"<text x='{F(x-12)}' y='354'>{tick}</text><path d='M{F(x)} 335 v4' stroke='black'/>");}for(int series=0;series<3;series++){var pts=new List<string>();for(int g=0;g<count;g++){if(g%Math.Max(1,count/200)!=0 && g!=count-1)continue;double value=series==0?Median(data.Select(r=>r.Item1[g])):series==1?Median(data.Select(r=>r.Item2[g])):data.Count(r=>!double.IsFinite(r.Item2[g]))/300.0;if(!double.IsFinite(value))continue;double y=series==2?335-290*value:45+290*Math.Clamp(-Math.Log10(Math.Max(value,1e-22))/22,0,1);pts.Add($"{F(left+width*(double)g/(count-1))} {F(y)}");}var color=series==0?"#276fbf":series==1?"#c14953":"#555555";sb.Append($"<polyline fill='none' stroke='{color}' points='{string.Join(" ",pts)}'/>");}sb.Append($"<text x='{left}' y='380' fill='#276fbf'>|mu| survivants</text><text x='{left+120}' y='380' fill='#c14953'>sigma survivants</text><text x='{left+250}' y='380' fill='#555'>échecs %</text>");panel++;}return sb.Append("</svg>").ToString();}
display(new PlotSvg(GradientFigure()));

aligné évaluations effectives 0 40000 log10 f -8 -4 0 4 8 10k 20k 30k <polyline fill='none' stroke='#276fbf' points='55.10 72.80 56.07 94.65 57.05 94.65 58.02 97.60 59.00 97.60 59.97 97.60 60.95 107.08 61.92 108.58 62.90 116.95 63.87 116.95 64.85 116.95 65.82 116.95 66.79 120.21 67.77 125.69 68.74 126.75 69.72 130.70 70.69 130.70 71.67 131.04 72.64 131.04 73.62 134.48 74.59 136.36 75.57 137.93 76.54 139.70 77.52 142.39 78.49 143.42 79.47 146.87 80.44 149.06 81.42 152.32 82.39 158.46 83.37 167.98 84.34 167.98 85.31 167.98 86.29 174.42 87.26 176.49 88.24 176.49 89.21 177.13 90.19 179.53 91.16 182.15 92.14 188.21 93.11 188.21 94.09 188.21 95.06 190.37 96.04 194.39 97.01 196.00 97.99 203.92 98.96 203.92 99.94 205.54 100.91 205.54 101.89 205.89 102.86 211.80 103.84 211.80 104.81 218.58 105.78 218.58 106.76 222.06 107.73 222.06 108.71 222.06 109.68 226.50 110.66 228.95 111.63 233.47 112.61 233.47 113.58 235.01 114.56 236.78 115.53 238.64 116.51 243.24 117.48 250.74 118.46 250.74 119.43 250.74 120.41 253.59 121.38 256.73 122.36 259.33 123.33 263.30 124.31 263.30 125.28 267.21 126.25 268.57 127.23 269.96 128.20 273.58 129.18 273.58 130.15 279.04 131.13 279.73 132.10 281.01 133.08 282.53 134.05 283.64 135.03 288.43 136.00 291.45 136.98 291.45 137.95 291.45 138.93 300.74 139.90 300.74 140.88 300.74 141.75 306.95'/> xNES SNES <polyline fill='none' stroke='#21856b' points='55.10 66.73 56.07 90.94 57.05 99.41 58.02 105.20 59.00 105.20 59.97 112.39 60.95 112.39 61.92 119.79 62.90 119.79 63.87 122.11 64.85 123.93 65.82 126.04 66.79 127.60 67.77 128.08 68.74 128.47 69.72 129.07 70.69 129.51 71.67 130.02 72.64 131.03 73.62 132.33 74.59 133.51 75.57 134.14 76.54 134.66 77.52 135.13 78.49 135.71 79.47 137.50 80.44 137.83 81.42 139.50 82.39 141.98 83.37 144.97 84.34 145.41 85.31 146.34 86.29 147.47 87.26 149.78 88.24 153.07 89.21 156.17 90.19 162.53 91.16 166.52 92.14 167.43 93.11 168.09 94.09 168.33 95.06 168.40 96.04 168.46 97.01 168.60 97.99 169.08 98.96 170.22 99.94 171.75 100.91 177.83 101.89 181.60 102.86 191.47 103.84 201.51 104.81 215.02 105.78 229.26 106.76 234.88 107.73 245.28 108.71 259.49 109.68 273.21 110.66 282.88 111.63 290.15 112.41 308.21'/> CMA-ES cible 1e-8 tourné évaluations effectives 0 40000 log10 f -8 -4 0 4 8 10k 20k 30k <polyline fill='none' stroke='#276fbf' points='495.10 69.64 496.07 89.30 497.05 96.06 498.02 96.50 499.00 96.75 499.97 98.89 500.95 107.86 501.92 111.39 502.90 111.39 503.87 111.53 504.85 111.53 505.82 119.41 506.79 123.68 507.77 123.75 508.74 131.38 509.72 133.03 510.69 133.03 511.67 135.76 512.64 139.52 513.62 142.84 514.59 152.47 515.57 152.47 516.54 152.47 517.52 153.78 518.49 154.01 519.47 156.29 520.44 158.85 521.42 158.88 522.39 164.12 523.37 169.94 524.34 169.94 525.31 172.44 526.29 172.44 527.26 177.32 528.24 184.51 529.21 184.51 530.19 184.51 531.16 191.48 532.14 191.48 533.11 191.48 534.09 193.60 535.06 198.11 536.04 198.11 537.01 199.34 537.99 200.66 538.96 201.71 539.94 205.11 540.91 207.26 541.89 209.73 542.86 213.20 543.84 213.20 544.81 219.45 545.78 224.80 546.76 224.80 547.73 224.80 548.71 227.08 549.68 227.08 550.66 227.08 551.63 232.45 552.61 236.81 553.58 236.81 554.56 238.53 555.53 240.33 556.51 242.80 557.48 246.57 558.46 253.20 559.43 253.20 560.41 253.20 561.38 258.15 562.36 258.15 563.33 262.33 564.31 265.20 565.28 267.78 566.25 268.14 567.23 268.14 568.20 269.41 569.18 274.91 570.15 281.70 571.13 281.70 572.10 281.82 573.08 284.80 574.05 284.80 575.03 290.90 576.00 291.53 576.98 296.99 577.95 300.26 578.93 300.26 579.90 301.18 580.88 303.06 581.27 309.36'/> xNES <polyline fill='none' stroke='#c14953' points='495.10 70.67 496.07 92.98 497.05 92.98 498.02 92.98 499.00 92.98 499.97 97.12 500.95 101.92 501.92 104.99 502.90 107.35 503.87 108.24 504.85 110.50 505.82 110.74 506.79 111.73 507.77 112.01 508.74 113.76 509.72 115.29 510.69 115.69 511.67 116.47 512.64 117.04 513.62 117.65 514.59 118.01 515.57 118.46 516.54 118.84 517.52 119.30 518.49 119.56 519.47 119.75 520.44 119.88 521.42 

gradient simple (1000 générations) log10 valeur échecs % génération 0 -5 -10 -15 -20 0 25 50 75 100 0 250 500 749 999 <polyline fill='none' stroke='#276fbf' points='65.00 45.09 66.85 45.67 68.70 46.27 70.56 46.84 72.41 47.43 74.26 48.00 76.11 48.59 77.96 49.15 79.81 49.72 81.67 50.33 83.52 50.91 85.37 51.41 87.22 52.03 89.07 52.59 90.93 53.16 92.78 53.79 94.63 54.36 96.48 54.96 98.33 55.49 100.19 56.13 102.04 56.70 103.89 57.28 105.74 57.95 107.59 58.55 109.44 59.05 111.30 59.56 113.15 60.24 115.00 60.80 116.85 61.46 118.70 61.97 120.56 62.53 122.41 63.10 124.26 63.68 126.11 64.30 127.96 64.83 129.81 65.42 131.67 66.00 133.52 66.64 135.37 67.22 137.22 67.80 139.07 68.42 140.93 68.98 142.78 69.55 144.63 70.19 146.48 70.69 148.33 71.34 150.19 71.89 152.04 72.52 153.89 73.10 155.74 73.73 157.59 74.30 159.44 74.90 161.30 75.54 163.15 76.00 165.00 76.60 166.85 77.21 168.70 77.68 170.56 78.36 172.41 78.99 174.26 79.49 176.11 80.03 177.96 80.57 179.81 81.16 181.67 81.69 183.52 82.32 185.37 82.88 187.22 83.47 189.07 84.11 190.93 84.62 192.78 85.17 194.63 85.78 196.48 86.28 198.33 86.86 200.19 87.47 202.04 88.12 203.89 88.58 205.74 89.19 207.59 89.81 209.44 90.33 211.30 90.81 213.15 91.47 215.00 92.03 216.85 92.50 218.70 93.11 220.56 93.61 222.41 94.25 224.26 94.82 226.11 95.48 227.96 96.06 229.81 96.60 231.67 97.28 233.52 97.95 235.37 98.48 237.22 99.16 239.07 99.67 240.93 100.31 242.78 100.93 244.63 101.36 246.48 102.01 248.33 102.66 250.19 103.18 252.04 103.86 253.89 104.40 255.74 105.05 257.59 105.63 259.44 106.19 261.30 106.74 263.15 107.31 265.00 107.75 266.85 108.33 268.70 108.99 270.56 109.55 272.41 110.13 274.26 110.73 276.11 111.22 277.96 111.77 279.81 112.45 281.67 113.04 283.52 113.65 285.37 114.26 287.22 114.83 289.07 115.42 290.93 115.99 292.78 116.69 294.63 117.09 296.48 117.68 298.33 118.32 300.19 118.98 302.04 119.52 303.89 120.07 305.74 120.68 307.59 121.32 309.44 121.80 311.30 122.34 313.15 122.91 315.00 123.52 316.85 124.05 318.70 124.70 320.56 125.26 322.41 125.84 324.26 126.32 326.11 126.97 327.96 127.45 329.81 128.02 331.67 128.60 333.52 129.11 335.37 129.63 337.22 130.31 339.07 130.88 340.93 131.41 342.78 132.23 344.63 132.77 346.48 133.31 348.33 133.95 350.19 134.56 352.04 135.00 353.89 135.50 355.74 136.15 357.59 136.83 359.44 137.32 361.30 137.94 363.15 138.45 365.00 139.13 366.85 139.65 368.70 140.07 370.56 140.58 372.41 141.13 374.26 141.76 376.11 142.50 377.96 142.98 379.81 143.71 381.67 144.28 383.52 144.84 385.37 145.49 387.22 146.10 389.07 146.57 390.93 147.08 392.78 147.67 394.63 148.26 396.48 148.82 398.33 149.39 400.19 149.90 402.04 150.50 403.89 151.18 405.74 151.66 407.59 152.30 409.44 152.95 411.30 153.56 413.15 153.99 415.00 154.64 416.85 155.27 418.70 155.89 420.56 156.39 422.41 156.98 424.26 157.54 426.11 158.24 427.96 158.79 429.81 159.45 431.67 160.07 433.52 160.59 435.00 161.10'/><polyline fill='none' stroke='#c14953' points='65.00 45.04 66.85 45.60 68.70 46.18 70.56 46.74 72.41 47.32 74.26 47.98 76.11 48.53 77.96 49.11 79.81 49.67 81.67 50.32 83.52 50.78 85.37 51.37 87.22 51.94 89.07 52.50 90.93 53.07 92.78 53.63 94.63 54.21 96.48 54.86 98.33 55.40 100.19 56.02 102.04 56.52 103.89 57.22 105.74 57.90 107.59 58.49 109.44 58.98 111.30 59.62 113.15 60.16 115.00 60.81 116.85 61.45 118.70 61.95 120.56 62.46 122.41 63.10 124.26 63.60 126.11 64.56 127.96 65.07 129.81 65.61 131.67 66.15 133.52 66.56 135.37 67.30 137.22 67.84 139.07 68.55 140.93 69.15 142.78 69.64 144.63 70.19 146.48 70.90 148.33 71.45 150.19 72.16 152.04 72.87 153.89 73.67 155.74 74.30 157.59 74.79 159.44 75.46 161.30 75.84 163.15 76.73 165.00 77.23 166.85 77.67 168.70 78.40 170.56 78.92 172.41 79.55 174.26 80.19 176.11 80.58 177.96 81.00 179.81 81.53 181.67 82.36 183.52 82.76 185.37 83.30 187.22 84.04 189.07 84.45 190.93 85.18 192.78 85.70 194.63 86.36 196.48 87.03 198.33 87.49 200.19 88.27 202.04 88.50 203.89 89.21 205.74 89.65 207.59 90.21 209.44 90.81 211.30 91.15 213.15 91.80 215.00 92.40 216.85 92.97 218.70 93.56 220.56 94.32 

Chaque courbe montre un run représentatif et le meilleur objectif cumulé. L'axe vertical est logarithmique ; la cible apparaît en pointillé.

## 3. Premier ordre et CMA-ES

On construit $\Sigma=A^TA$ anisotrope et l'on classe les échantillons selon leur fitness. Avec $G=\sum_k u_k(s_ks_k^T-I)$ et $U=\sum_k u_k$, on compare $A^T\exp(\eta G)A$ à $(1-U\eta)\Sigma+\eta\sum_k u_k(A^Ts_k)(A^Ts_k)^T$. Leur différence est du second ordre, de coefficient $\|A^TG^2A\|_F/2$. Ce rang-$\mu$ isolé n'est pas le CMA-ES complet, qui inclut des chemins d'évolution.

In [11]:
double[,] Transpose(double[,] a){int rows=a.GetLength(0),cols=a.GetLength(1);var t=new double[cols,rows];for(int i=0;i<rows;i++)for(int j=0;j<cols;j++)t[j,i]=a[i,j];return t;}
double Norm(double[,] a){double sum=0;foreach(double v in a)sum+=v*v;return Math.Sqrt(sum);}
var rng5=new Random(42);int d5=5,n5=Lambda(d5);var w5=Utilities(n5);var Q5=Rotation(d5,42);var A5=new double[d5,d5];for(int i=0;i<d5;i++)for(int j=0;j<d5;j++)A5[i,j]=Q5[i,j]*(.5+1.5*j/(d5-1));var At5=Transpose(A5);var sigma5=M.Multiply(At5,A5);
var samples5=Enumerable.Range(0,n5).Select(_=>Enumerable.Range(0,d5).Select(_=>Normal(rng5)).ToArray()).OrderBy(z=>{var y=M.Multiply(At5,z);return y.Select((v,i)=>Math.Pow(1e6,i/9.0)*v*v).Sum();}).ToArray();
var G=new double[d5,d5];for(int k=0;k<n5;k++)for(int i=0;i<d5;i++)for(int j=0;j<d5;j++)G[i,j]+=w5[k]*(samples5[k][i]*samples5[k][j]-(i==j?1:0));double U=w5.Sum();var predicted=Norm(M.Multiply(M.Multiply(At5,M.Multiply(G,G)),A5))/2;Console.WriteLine($"U={U:G4}, coefficient quadratique transporté={predicted:G6}; diagonale Sigma=[{string.Join(", ",Enumerable.Range(0,d5).Select(i=>sigma5[i,i].ToString("F3")))}]");
double smallRatio=double.NaN;foreach(double rate in new[]{.5,.1,.02,.004}){var arg=new double[d5,d5];for(int i=0;i<d5;i++)for(int j=0;j<d5;j++)arg[i,j]=rate*G[i,j];var xnesCov=M.Multiply(M.Multiply(At5,M.MatrixExponential(arg)),A5);var rankCov=new double[d5,d5];for(int i=0;i<d5;i++)for(int j=0;j<d5;j++)rankCov[i,j]=(1-U*rate)*sigma5[i,j];for(int k=0;k<n5;k++){var y=M.Multiply(At5,samples5[k]);for(int i=0;i<d5;i++)for(int j=0;j<d5;j++)rankCov[i,j]+=rate*w5[k]*y[i]*y[j];}var delta=new double[d5,d5];for(int i=0;i<d5;i++)for(int j=0;j<d5;j++)delta[i,j]=xnesCov[i,j]-rankCov[i,j];double error=Norm(delta),ratio=error/(rate*rate);if(rate==.004)smallRatio=ratio;Console.WriteLine($"eta={rate}: erreur covariance={error:G6}, erreur/eta²={ratio:G6}");}
if(Math.Abs(smallRatio-predicted)>Math.Max(1e-8,.03*predicted))throw new Exception("Coefficient transporté : convergence insuffisante au petit pas");
var rotationExpm=M.MatrixExponential(new double[,]{{0,1},{-1,0}});if(Math.Abs(rotationExpm[0,0]-Math.Cos(1))+Math.Abs(rotationExpm[0,1]-Math.Sin(1))>1e-12)throw new Exception("Oracle expm rotation");
// Oracle fermé anisotrope indépendant : G²=2I, A=diag(2,1).
var testG=new double[,]{{1,1},{1,-1}};double t=.004,root=Math.Sqrt(2);var testArg=new double[,]{{t,t},{t,-t}};var expTest=M.MatrixExponential(testArg);double closedError=0;for(int i=0;i<2;i++)for(int j=0;j<2;j++){double closed=(i==j?Math.Cosh(root*t):0)+Math.Sinh(root*t)/root*testG[i,j];closedError=Math.Max(closedError,Math.Abs(expTest[i,j]-closed));}var oracleA=new double[,]{{2,0},{0,1}};double oracleCoefficient=Norm(M.Multiply(M.Multiply(Transpose(oracleA),M.Multiply(testG,testG)),oracleA))/2;if(closedError>1e-12 || Math.Abs(oracleCoefficient-Math.Sqrt(17))>1e-12)throw new Exception("Oracle fermé anisotrope");Console.WriteLine($"Oracle expm fermé : écart={closedError:G3}; coefficient anisotrope indépendant={oracleCoefficient:G6}");

U=-5.551E-17, coefficient quadratique transporté=2.39922; diagonale Sigma=[0.250, 0.766, 1.562, 2.641, 4.000]


eta=0.5: erreur covariance=0.48105, erreur/eta²=1.9242


eta=0.1: erreur covariance=0.0229065, erreur/eta²=2.29065


eta=0.02: erreur covariance=0.000950754, erreur/eta²=2.37688


eta=0.004: erreur covariance=3.83156E-05, erreur/eta²=2.39472


Oracle expm fermé : écart=0; coefficient anisotrope indépendant=4.12311


Le rapport de l'erreur à $\eta^2$ se rapproche du coefficient transporté calculé. Le contrôle fermé à deux dimensions teste séparément l'exponentielle ; l'écart au petit pas autorise le terme d'ordre supérieur. La formule rang-$\mu$ seule ne reproduit pas l'ensemble de CMA-ES.

### Exercice 3 — Rosenbrock tourné

Implémenter Rosenbrock, translater son optimum puis appliquer la rotation. Comparer les trois méthodes à budget égal.

In [12]:
double? Rosenbrock(double[] x) { // TODO : somme 100(x[i+1]-x[i]^2)^2+(1-x[i])^2
 return null;
}
double[] RotateRosenbrockInput(double[] x, double[,] q) { // TODO : appliquer q au vecteur, puis ajouter 1 pour translater l'optimum
 return null;
}
var trial=Rosenbrock(new[]{1d,1d}); var translated=RotateRosenbrockInput(new[]{0d,0d},M.Identity(2));
if(trial==null || translated==null) Console.WriteLine("Exercice 3 à compléter : Rosenbrock et transformation tournée");
else {var q=Rotation(2,2014);var transformed=RotateRosenbrockInput(new[]{.3,-.4},q);double[] expected={1+q[0,0]*.3-q[0,1]*.4,1+q[1,0]*.3-q[1,1]*.4};bool valid=trial.HasValue && Math.Abs(trial.Value)<1e-12 && Math.Abs(Rosenbrock(new[]{0d,0d}).GetValueOrDefault(double.NaN)-1)<1e-12 && Math.Abs(Rosenbrock(new[]{-1d,1d}).GetValueOrDefault(double.NaN)-4)<1e-12 && translated.Length==2 && translated.All(v=>Math.Abs(v-1)<1e-12) && transformed!=null && transformed.Length==2 && transformed.Zip(expected,(a,b)=>Math.Abs(a-b)<1e-12).All(v=>v);Console.WriteLine($"Fixtures Rosenbrock et transformation : {valid}");if(!valid)throw new Exception("Corriger Rosenbrock ou sa rotation");}

Exercice 3 à compléter : Rosenbrock et transformation tournée


## À retenir

| Question | Réponse |
|---|---|
| Pourquoi le gradient naturel ? | La géométrie de la distribution modifie l'échelle du pas. |
| Que préserve le classement ? | Toute transformation strictement croissante de l'objectif. |
| Que révèle une rotation ? | La sensibilité d'une covariance diagonale aux axes du problème. |
| Qu'apporte le développement ? | Le lien local entre exponentielle matricielle et mise à jour rang-$\mu$. |

## Vérifiez votre compréhension

Pourquoi une mise à jour additive de $\sigma$ peut-elle échouer ?
<details><summary>Réponse</summary>Elle peut rendre l'écart-type négatif ou non fini ; la distribution n'est alors plus valide.</details>

## Pour aller plus loin

Wierstra et al., *Natural Evolution Strategies*, JMLR 2014 ; [version Python](Search-05b-NaturalEvolutionStrategies.ipynb).